# Варіант 1: Інтернет-магазин (Обробка даних клієнтів та інженерія ознак)
Практичне заняття: Похідні змінні, дискретизація (cut/qcut), масштабування (min-max, z-score) та one-hot кодування (get_dummies)

### Завдання 1. Побудова набору клієнтів (10 замовлень)

In [ ]:
import pandas as pd
import numpy as np

# Фіксуємо базову ціну товару для Варіанта 1 (наприклад, 250 грн)
BASE_PRICE = 250

# Створення DataFrame з 10 замовленнями
data = [
    {"вік": 22, "дохід": 18000, "кількість": 2, "місто_доставки": "Київ"},
    {"вік": 45, "дохід": 42000, "кількість": 5, "місто_доставки": "Львів"},
    {"вік": 31, "дохід": 28000, "кількість": 1, "місто_доставки": "Одеса"},
    {"вік": 58, "дохід": 65000, "кількість": 3, "місто_доставки": "Харків"},
    {"вік": 25, "дохід": 22000, "кількість": 4, "місто_доставки": "Київ"},
    {"вік": 62, "дохід": 35000, "кількість": 2, "місто_доставки": "Львів"},
    {"вік": 19, "дохід": 15000, "кількість": 1, "місто_доставки": "Одеса"},
    {"вік": 39, "дохід": 38000, "кількість": 6, "місто_доставки": "Київ"},
    {"вік": 50, "дохід": 52000, "кількість": 3, "місто_доставки": "Харків"},
    {"вік": 28, "дохід": 26000, "кількість": 2, "місто_доставки": "Львів"}
]

df = pd.DataFrame(data)

print("--- Початковий датасет клієнтів ---")
display(df)

### Завдання 2. Розрахунок похідних змінних

In [ ]:
# 1. Розрахунок суми замовлення та частки доходу
df["сума_замовлення"] = BASE_PRICE * df["кількість"]
df["частка_доходу"] = df["сума_замовлення"] / df["дохід"]

display(df[["вік", "дохід", "кількість", "сума_замовлення", "частка_доходу"]])

**Аналітична цінність похідної змінної `частка_доходу`:**
Спеціально розрахована змінна `частка_доходу` відображає **фінансове навантаження замовлення на бюджет клієнта** (відносну купівельну спроможність та готовність витрачати). 
- Самі по собі `сума_замовлення` (наприклад, 1500 грн) і `дохід` (65 000 грн проти 15 000 грн) не показують, наскільки чутливою є ця покупка для клієнта.
- Клієнт з доходом 15 000 грн, що робить замовлення на 1000 грн, віддає **6.67%** свого місячного доходу, тоді як клієнт з доходом 65 000 грн при замовленні на 1500 грн віддає лише **2.31%**. 
- Отже, `частка_доходу` дозволяє оцінити цінову чутливість, схильність до ризику та лояльність покупця набагато ефективніше, ніж абсолютні величини доходу чи чека окремо.

### Завдання 3. Дискретизація (pd.cut проти pd.qcut)

In [ ]:
# 1. pd.cut — розбиття на 3 інтервали однакової ширини (діапазону)
df["вік_група_cut"] = pd.cut(df["вік"], bins=3)

# 2. pd.qcut — розбиття на 3 квантилі з однаковою кількістю спостережень
df["вік_група_qcut"] = pd.qcut(df["вік"], q=3)

print("--- Частоти категорій pd.cut() (рівна ширина інтервалу) ---")
print(df["вік_група_cut"].value_counts().sort_index())

print("\n--- Частоти категорій pd.qcut() (рівна кількість спостережень) ---")
print(df["вік_група_qcut"].value_counts().sort_index())

**Порівняння та пояснення відмінності cut і qcut:**
- **`pd.cut()`** ділить увесь числовий діапазон значень змінної (від $min$ до $max$) на 3 рівні за шириною квантувальні відрізки: $\Delta = (max - min) / 3$. Через це кількість клієнтів, що потрапляють у кожен інтервал, залежить від щільності розподілу даних і може бути дуже нерівномірною (наприклад, 4, 3, 3 або 5, 2, 3).
- **`pd.qcut()`** ділить вибірку на основі процентилів (квантилів) так, щоб у кожен інтервал потрапила **однакова (або максимально близька) кількість об'єктів** ($\approx 1/3$ від усіх рядків). Для цього межі самих інтервалів адаптуються під щільність даних і стають нерівними за шириною.

### Завдання 4. Масштабування (Min-Max Нормалізація та Z-Score Стандартизація)

In [ ]:
# 1. Min-Max normalization в інтервал [0, 1]
df["вік_minmax"] = (df["вік"] - df["вік"].min()) / (df["вік"].max() - df["вік"].min())
df["дохід_minmax"] = (df["дохід"] - df["дохід"].min()) / (df["дохід"].max() - df["дохід"].min())

# 2. Z-score standardization (середнє = 0, std = 1)
df["вік_zscore"] = (df["вік"] - df["вік"].mean()) / df["вік"].std()
df["дохід_zscore"] = (df["дохід"] - df["дохід"].mean()) / df["дохід"].std()

print("--- Порівняння оригінальних та масштабованих змінних (перші 5 рядків) ---")
cols_to_show = [
    "вік", "вік_minmax", "вік_zscore",
    "дохід", "дохід_minmax", "дохід_zscore"
]
display(df[cols_to_show].head())

**Обґрунтування потреби в масштабуванні:**
Немасштабовані змінні `дохід` (порядок величин в десятки тисяч грн: 15 000 – 65 000) та `вік` (порядок величин в одиниці/десятки років: 18 – 62) мають **неспівмірні масштаби та одиниці вимірювання**. 
- Якщо застосувати алгоритми, чутливі до відстаней (наприклад, k-NN, k-Means, евклідова відстань) або регресію з градієнтним спуском чи L1/L2-регуляризацією, змінна `дохід` буде домінувати у розрахунках відстаней за рахунок великої машинної амплітуди. 
- Внесок `віку` стане математично непомітним (нехтовно малим), що призведе до спотворення результатів моделювання. Масштабування приводить обидва параметри до єдиної числової шкали.

### Завдання 5. Індикаторне (One-Hot) кодування

In [ ]:
# 1. One-Hot кодування для міста доставки
city_dummies = pd.get_dummies(df["місто_доставки"], prefix="місто", dtype=int)

# 2. Об'єднання з основною таблицею
df_final = pd.concat([df, city_dummies], axis=1)

print(f"Кількість унікальних міст у вибірці: {df['місто_доставки'].nunique()}")
print(f"Кількість згенерованих One-Hot стовпчиків: {city_dummies.shape[1]}")

print("\n--- Результуюча таблиця з індикаторними стовпчиками (перші 5 рядків) ---")
display(df_final.head())

**Обґрунтування використання One-Hot кодування замість порядкового:**
Змінна `місто_доставки` належить до **номінальної шкали вимірювання**. Між містами (Київ, Львів, Одеса, Харків) не існує жодного природного математичного порядку, ієрархії чи відношення «більше/менше».
- Якщо б ми присвоїли містам числа 1, 2, 3, 4 (порядкове кодування), математичні моделі сприйняли б це як відношення порядку (наприклад, що Харків (4) у 4 рази більший або «важливіший» за Київ (1), або що середня відстань між якимись містами дорівнює 2.5). 
- **One-Hot кодування (pd.get_dummies)** створює окремі незалежні бінарні індикатори (0 або 1) для кожного міста, усуваючи хибні неіснуючі зв'язки між категоріями.

### Відповіді на контрольні питання:

**1. Чи можете пояснити різницю між pd.cut() і pd.qcut() — коли інтервали однакової довжини, а коли гарантована однакова кількість спостережень?**
- **`pd.cut()`** формує інтервали **однакової довжини (ширини діапазону)** $\Delta x = \text{const}$. Кількість об'єктів у кожному інтервалі залежить від фактичного розподілу даних і зазвичай відрізняється.
- **`pd.qcut()`** формує інтервали на основі квантилів, що гарантують **однакову (або максимально близьку) кількість спостережень** $N = \text{const}$ у кожній групі. При цьому межі інтервалів виходять нерівними.

**2. Чи розумієте, чому для номінальної змінної (місто) застосовують one-hot кодування, а не порядкове чи просто числове (1, 2, 3)?**
Номінальні змінні не мають порядку. Числове кодування ($1, 2, 3...$) вносить у дані штучний впорядкований вектор та метрику відстані, змушуючи алгоритми машинного навчання вважати категорію $3$ більшою за $1$. One-Hot кодування перетворює $K$ категорій у $K$ незалежних ортогональних бінарних векторів (стовпчиків $0/1$), що зберігає якісну рівноправність категорій.

**3. Чи можете пояснити, навіщо взагалі масштабувати змінні перед методом, чутливим до масштабу (дистанційні методи, регресія з регуляризацією)?**
Якщо змінні мають різні масштаби (наприклад, дохід $10^4$ і вік $10^1$), то у метричних алгоритмах (k-NN, k-Means, SVM) евклідова відстань буде на $99.9\%$ визначатися лише змінною з більшим числовим діапазоном. У регресіях із регуляризацією (Lasso, Ridge) штрафи на коефіцієнти будуть несправедливо затискати ознаки з меншим масштабом. Масштабування нормалізує ваги й відстані, роблячи внесок усіх ознак співмірним.

**4. Чи можете пояснити, чому похідна змінна (наприклад, частка доходу на замовлення) часто несе більше сенсу для аналізу, ніж вихідні складники окремо?**
Вихідні складники відображають лише ізольовані факти (абсолютний розмір чека чи абсолютний дохід). Похідна змінна **комбінує їх у відносну величину**, що відображає якісно новий бізнес-зміст — наприклад, фінансове навантаження на клієнта, ризиковість покупки або відносну чутливість до ціни. Це дозволяє порівнювати між собою клієнтів із кардинально різними рівнями доходів та чеків у єдиній відносній шкалі.